# Regression Model Comparison

Put `train.csv` and `test.csv` in the same directory as this notebook.

- Set `TARGET_COLUMN` to your target column.
- No preprocessing is performed.
- K-fold CV and hyperparameter tuning use **train.csv only**.
- `test.csv` is used only for final evaluation.
- The final cell reports **R², MAE and RMSE** for every model.


In [ ]:
# Common imports and data loading
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, RandomizedSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

TARGET_COLUMN = "target"   # <-- CHANGE THIS

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

X = train.drop(columns=[TARGET_COLUMN])
y = train[TARGET_COLUMN]

X_test = test.drop(columns=[TARGET_COLUMN])
y_test = test[TARGET_COLUMN]

CV = KFold(n_splits=2, shuffle=True, random_state=0)

print("Train shape:", train.shape)
print("Test shape :", test.shape)


## 1. Simple Linear Regression


In [ ]:
from sklearn.linear_model import LinearRegression

regressor = LinearRegression()

# Simple linear regression uses ONE input feature.
# fit_intercept=True estimates the y-axis intercept.
# Cross-validation checks generalization using train.csv only.
cv_scores = cross_val_score(
    regressor, X.iloc[:, [0]], y,
    cv=CV, scoring="r2"
)

regressor.fit(X.iloc[:, [0]], y)

print("Mean 5-Fold CV R2:", cv_scores.mean())
print("CV R2 scores:", cv_scores)


## 2. Multiple Linear Regression


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV

regressor = LinearRegression()

# fit_intercept: whether to estimate the intercept.
# positive: if True, forces all regression coefficients to be non-negative.
param_grid = {
    "fit_intercept": [True, False],
    "positive": [False, True]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


## 3. Polynomial Regression


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV

regressor = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("linear", LinearRegression())
])

# degree: controls polynomial complexity and interaction terms.
# include_bias=False: avoids an extra constant feature because LinearRegression has an intercept.
param_grid = {
    "poly__degree": [2, 3],
    "linear__fit_intercept": [True, False]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


## 4. Support Vector Regression (SVR)


In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV

regressor = SVR(
    kernel="poly",
    C=100,
    epsilon=0.1,
    gamma="scale",
    degree=3,
    coef0=1.0
)

# kernel: defines the function used to model non-linear relationships.
#         Using "poly" instead of "rbf" here (rbf was too slow on this dataset).
# C: controls the penalty for training errors; larger values fit more strictly.
# epsilon: defines the error tolerance tube.
# gamma: controls the influence range of points for the poly/rbf kernels.
# degree: degree of the polynomial kernel (only used when kernel="poly").
# coef0: independent term in the poly kernel; shifts how much high- vs low-degree terms matter.
param_grid = {
    "kernel": ["poly", "linear"],
    "C": [1, 10, 100],
    "epsilon": [0.01, 0.1, 0.2],
    "gamma": ["scale", "auto"],
    "degree": [2, 3],
    "coef0": [0.0, 1.0]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


## 5. Decision Tree Regression


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import RandomizedSearchCV

regressor = DecisionTreeRegressor(
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=0
)

# max_depth: limits tree depth and model complexity.
# min_samples_split: minimum samples required before splitting a node.
# min_samples_leaf: minimum samples allowed in each leaf; larger values can reduce overfitting.
param_grid = {
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


## 6. Random Forest Regression


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

regressor = RandomForestRegressor(
    n_estimators=10,
    random_state=0
)

# n_estimators: number of trees in the forest.
# max_depth: maximum depth of each tree; controls complexity.
# min_samples_split: minimum samples required to split a node.
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


## 7. XGBoost Regression


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

regressor = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=1.0,
    random_state=0,
    objective="reg:squarederror"
)

# n_estimators: number of boosting trees.
# learning_rate: size of each boosting step.
# max_depth: maximum depth of each tree; controls complexity.
# subsample: fraction of training rows used by each tree; can reduce overfitting.
param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5],
    "subsample": [0.8, 1.0]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


## 8. LightGBM Regression


In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV

regressor = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=-1,
    num_leaves=31,
    random_state=0,
    verbosity=-1
)

# n_estimators: number of boosting iterations.
# learning_rate: contribution of each boosting tree.
# num_leaves: controls tree complexity; larger values can model more complex patterns.
# max_depth: limits tree depth; -1 means no explicit limit.
param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 10]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


## 9. CatBoost Regression


In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import RandomizedSearchCV

regressor = CatBoostRegressor(
    iterations=200,
    learning_rate=0.1,
    depth=6,
    loss_function="RMSE",
    verbose=False,
    random_seed=0
)

# iterations: number of boosting rounds.
# learning_rate: contribution of each boosting round.
# depth: depth of the trees; higher values increase model complexity.
param_grid = {
    "iterations": [100, 200],
    "learning_rate": [0.05, 0.1],
    "depth": [4, 6, 8]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


## Final Test-Set Evaluation


In [ ]:
# IMPORTANT:
# Every model is tuned using train.csv only.
# The best configuration is then refit on all of train.csv.
# test.csv remains completely untouched until this cell.

models = {
    "Simple Linear Regression": RandomizedSearchCV(
        LinearRegression(),
        {"fit_intercept": [True, False]},
        cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
    ),

    "Multiple Linear Regression": RandomizedSearchCV(
        LinearRegression(),
        {"fit_intercept": [True, False], "positive": [False, True]},
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "Polynomial Regression": RandomizedSearchCV(
        Pipeline([
            ("poly", PolynomialFeatures(include_bias=False)),
            ("linear", LinearRegression())
        ]),
        {
            "poly__degree": [2, 3],
            "linear__fit_intercept": [True, False]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "SVR": RandomizedSearchCV(
        SVR(),
        {
            "kernel": ["poly", "linear"],
            "C": [1, 10, 100],
            "epsilon": [0.01, 0.1, 0.2],
            "gamma": ["scale", "auto"],
            "degree": [2, 3],
            "coef0": [0.0, 1.0]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "Decision Tree Regression": RandomizedSearchCV(
        DecisionTreeRegressor(random_state=0),
        {
            "max_depth": [None, 5, 10, 20],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "Random Forest Regression": RandomizedSearchCV(
        RandomForestRegressor(random_state=0),
        {
            "n_estimators": [100, 200],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "XGBoost Regression": RandomizedSearchCV(
        XGBRegressor(
            random_state=0,
            objective="reg:squarederror"
        ),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 5],
            "subsample": [0.8, 1.0]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "LightGBM Regression": RandomizedSearchCV(
        LGBMRegressor(random_state=0, verbosity=-1),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "num_leaves": [15, 31, 63],
            "max_depth": [-1, 10]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "CatBoost Regression": RandomizedSearchCV(
        CatBoostRegressor(
            loss_function="RMSE",
            verbose=False,
            random_seed=0
        ),
        {
            "iterations": [100, 200],
            "learning_rate": [0.05, 0.1],
            "depth": [4, 6, 8]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    )
}

results = []

for name, model in models.items():

    # Simple Linear Regression uses only the first feature.
    if name == "Simple Linear Regression":
        model.fit(X.iloc[:, [0]], y)
        predictions = model.predict(X_test.iloc[:, [0]])
    else:
        model.fit(X, y)
        predictions = model.predict(X_test)

    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    results.append([
        name,
        r2,
        mae,
        rmse,
        model.best_params_
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "R2 Score",
        "MAE",
        "RMSE",
        "Best Hyperparameters"
    ]
)

results_df = results_df.sort_values(
    "R2 Score",
    ascending=False
).reset_index(drop=True)

display(results_df)

best_row = results_df.iloc[0]

print("Best model:", best_row["Model"])
print("Best test R2:", best_row["R2 Score"])
print("Selected Hyperparameters:")
print(best_row["Best Hyperparameters"])


## Save Best Regression Model

The best regression model is selected using **CV R²** and saved as a `.pkl` file.
The saved object contains the fitted model, feature columns, target column, and task metadata.


In [ ]:
# ============================================================
# SAVE BEST REGRESSION MODEL
# ============================================================
# The winner is selected using the cross-validation R² score.
# The fitted best_estimator_ has already been refit on the training data.

import joblib
from IPython.display import display, FileLink

best_model_name = best_row["Model"]
best_model = models[best_model_name].best_estimator_

regression_model_package = {
    "model": best_model,
    "feature_columns": list(X.columns) if best_model_name != "Simple Linear Regression" else [X.columns[0]],
    "target_column": TARGET_COLUMN,
    "task": "regression",
    "selection_metric": "CV R²",
    "best_cv_score": float(
        models[best_model_name].best_score_
    )
}

regression_pkl_path = "best_regression_model.pkl"

joblib.dump(
    regression_model_package,
    regression_pkl_path
)

print("=" * 70)
print("Best regression model saved successfully.")
print("Model:", best_model_name)
print("File:", regression_pkl_path)
print("=" * 70)

display(FileLink(regression_pkl_path, result_html_prefix="⬇️ Download Best Regression Model: "))
